<a href="https://colab.research.google.com/github/redinbluesky/nlp-with-transformers/blob/main/08-효율적인_트랜스포머_구축.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 0 개요](#chapter0)
* [Chapter 1 의도 탐지 예제](#chapter1)
* [Chapter 2 벤치마크 클래스 만들기](#chapter2)

## Chapter 0 개요 <a class="anchor" id="chapter0"></a>

1. 모델이 너무 느리거나 크다면, 최고 성능의 모델이라도 유용하지 않다.
    - 빠르고 작은 모델을 구축하는 방법이 있다.
    - 모델의 용량을 줄이면 종종 성능이 저하되는데 이를 최소화하는 방법이 있다.
    - 지식정체, 양자화, 가지치기, ONNX포맷, ONNX런타임을 사용한 그래프최적화가 있다.

2. 로블록스 엔지니어링 팀은 블로그에 이러한 기법을 사용하여 대규모 트랜스포머 모델을 최적화하는 방법을 설명했다.
    -  지식 정제와 양자화를 연결해 레이턴시와 BERT 분류기의 성능을 30배 향상시켰다.

        ![Optimizing Large Transformer Models at Roblox](image/08-01-optimizing-large-transformer-models-at-roblox.png)

3. 각 기술의 장점과 단점을 이해하기 위해 의도 참지 예제를 사용하여 이러한 기법을 실험해본다.


## Chapter 1 의도 탐지 예제 <a class="anchor" id="chapter0"></a>
1. 고객이 다음과 같은 메시지를 보냈다고 가정한다.
    - "Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th In Paris and I need a 15 passanger van."

2. 의도 분류기는 이를 자동으로 Car Rent로 분류하고 응답한다.
    - 고객이 사전에 저으이된 의도에 속하지 않은 쿼리를 제공하면 시스템은 대체 응답을 출력해댜한다.
    - 아래의 그림은 범위 안에 없는 질문을 하자 잘 못된 응답을 출력했고, 세 번재 질문에서 카테고리에 없는 응답이라는 것을 인지하고 적절한 응답을 출력했다.

        ![Intent Detection Example](image/08-02-intent-detection-example.png)

3. CLINC150 데이터셋에서 미세튜닝해 약 94%의 정확도를 달성한 BERT 베이스 모델을 기준모델로 사용한다.
    - 이 데이터셋에는 150개의 의도와 은행, 여행 등 10개 분야로 분류된 22,500개 쿼리가 포함되어 있다.
    - 범위를 벗어난 쿼리가 1,200개 있고 이런 쿼리는 의도 클래스 oos에 속한다.
    - 실전에서는 회사 내부 데이터셋도 수집하겟지만, 공개 데이터를 사용하는 것이 빠르게 반복하고 초기 결과를 생성하기 좋다.

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
from transformers import pipeline

bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: transformersbook/bert-base-uncased-finetuned-clinc
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# 쿼리를 전달해 모델로부터 예측한 의도와 신뢰점수를 얻는다.
query = "Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th In Paris and I need a 15 passanger van."
pipe(query)

[{'label': 'car_rental', 'score': 0.5804867148399353}]

## Chapter 2 벤치마크 클래스 만들기 <a class="anchor" id="chapter2"></a>
1. 애플리케이션의 중요 지표가 이미정의되어 있다고 가정하고 모델의 지표를 최적화하는 데 집중한다.

2. 모델 성능이 중요한 경우는 다음과 같다.
   - 오류가 발생했을 때 손실 비용이 큰 상황(또는 오류 발생을 줄이기 위해 사람의 참여가 최선일 때)
   - 모델의 지표가 조금 향상되면 전체적으로 큰 이득을 얻을 수 있는 상황 

3. 레이턴시가 중요한 경우
    - 대량의 트래픽을 처리하는 실시간 환경에서 고려한다.

4. 메모리가 중요한 경우
    - 메모리가 모바일과 에지 장치에서 제한적일 때 고려한다.
    

In [4]:
# 파이프라인과 테스트 세트가 주어지면 성능을 측정하는 간단한 벤치마크 클래스를 만든다.
class PerformanceBenchmark:
    """
    optim_type: str
        여러가지 최적화 기법의 성능을 비교하기 위한 문자열 식별자
    """
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        # 나중에 정의한다.
        pass

    def compute_size(self):
        # 나중에 정의한다.
        pass

    def time_pipeline(self):
        # 나중에 정의한다.
        pass

    def run_benchmark(self):
        """딕셔너리에 optim_type을 키로 하여 성능 메트릭을 저장한다."""
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics

In [5]:
# 기준 모델을 미세 튜닝하는 데 사용한 CLINC150 데이터셋을 불러온다.
from datasets import load_dataset

# "plus": 범위 밖의 훈ㅁ련 샘플이 담긴 서브셋을 의미한다.
# "clinc_oos" 데이터셋에는 150개의 의도 클래스가 있다.
clinc = load_dataset("clinc_oos", "plus")

In [6]:
# 테스트 샘플 하나를 살표본다.
sample = clinc["test"][42]
print(f"샘플 쿼리: {sample}")

샘플 쿼리: {'text': 'transfer $100 from my checking to saving account', 'intent': 133}


In [7]:
# 의도는 ID로 제공되지만 features 속성을 사용하면 문자열로 매핑된다.
intents = clinc["test"].features["intent"]
print(f"샘플 의도 ID: {sample['intent']}, 의도 이름: {intents.int2str(sample['intent'])}")

샘플 의도 ID: 133, 의도 이름: transfer


In [8]:
# commute_accuracy() 메서드를 구현한다.
import evaluate

accuracy_score = evaluate.load("accuracy")

In [9]:
def compute_accuracy(self):
    """
    정확한 정수 지표는 예측과 정답을 기대한다.
    파이프라인을 사용해 text 필드에서 예측을 추출하고, intent 객체의 str2int()를 사용해 각 예측을 해당 ID로 변환한다.
    """
    preds, labels = [], []
    for example in self.dataset:
        pred = self.pipeline(example["text"])[0]["label"]
        label = example["intent"]
        preds.append(intents.str2int(pred))
        labels.append(label)
    accuracy = accuracy_score.compute(predictions=preds, references=labels)
    print(f"정확도: {accuracy['accuracy']:.3f}")
    return accuracy

PerformanceBenchmark.compute_accuracy = compute_accuracy

5. 파이토치의 torch.save() 함수를 사용해 모델을 디스크에 직렬화하고 크기를 계산한다.
    - 파이토치에서 모델을 저장할 때 state_dict() 메서드를 사용해 모델의 매개변수를 저장한다.
    - 각각의 키/값 쌍이 BERT의 층과 텍션에 해당한다.

In [10]:
import torch

list(pipe.model.state_dict().items())[42]
torch.save(pipe.model.state_dict(), "model.pt")

In [11]:
import torch
from pathlib import Path

def compute_size(self):
    """
    Path.stat() 메서드를 사용해 모델 파일의 크기를 바이트 단위로 계산한다.
    """
    state_dic = self.pipeline.model.state_dict()
    tmp_path = Path("model.pt")
    torch.save(state_dic, tmp_path)
    # 메가바이트 단위로 크기를 계산한다.
    size_mb = tmp_path.stat().st_size / (1024 * 1024)
    print(f"모델 크기: {size_mb:.2f} MB")
    # 임시 파일을 삭제한다.
    tmp_path.unlink()
    return {"size_mb": size_mb}

PerformanceBenchmark.compute_size = compute_size

6. 쿼리마다 평균적인 레이턴시를 재기 위해 time_pipeline() 메서드를 구현한다.
    - 파이프라인에 텍스트 쿼리를 주입하고 모델로부터 예측된 의도가 반환되기까지 걸리는 시간을 측정한다.

In [13]:
# 파이프라인데 테스트 쿼리를 전달하고 perf_count()를 사용하여 코드 실행의 시작과 끝 시간 차이를 밀리초 단위로 계산한다.
from time import perf_counter

for _ in range(3):
    start_time = perf_counter()
    _ = pipe(query)
    latency = perf_counter() - start_time
    print(f"레이턴시: {1000 * latency:.3f} ms")


레이턴시: 45.422 ms
레이턴시: 43.121 ms
레이턴시: 36.947 ms


7. 결과를 보면 레이턴시의 차이가 크다.
    - 파이프라인을 여러 번 실행해서 레이턴시를 수집하고 그 결과를 표준 편차를 계산한 후, 분포를 시각화한다.

In [16]:
import numpy as np

def time_pipeline(self, query="What is the pin number for my account?"):
    """
    레이턴시는 쿼리의 길이마다 달라지므로 동일한 query를 사용해 모델을 벤치마킹한다.
    """
    latencies = []
    # 워밍업
    for _ in range(10):
        _ = self.pipeline(query)
    # 실제 측정
    for _ in range(100):
        start_time = perf_counter()
        _ = self.pipeline(query)
        latency = perf_counter() - start_time
        latencies.append(latency)  

    # 통계 계산
    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    print(f"평균 레이턴시 (ms): {time_avg_ms:.2f} ± {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

PerformanceBenchmark.time_pipeline = time_pipeline

In [17]:
# BERT 기준 모델의 성능을 벤치마킹한다.
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type="BERT baseline")
perf_metrics = pb.run_benchmark()
perf_metrics

모델 크기: 418.15 MB


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


평균 레이턴시 (ms): 5.11 ± 4.12
정확도: 0.867


{'BERT baseline': {'size_mb': 418.15016078948975,
  'time_avg_ms': np.float64(5.106250350015671),
  'time_std_ms': np.float64(4.124923070319786),
  'accuracy': 0.8672727272727273}}

8. 평균 레이턴시 값은 사용하는 하드웨어 종료에 따라 달라진다.
    - GPU에서 처리하면 배치 처리가 가능해 성능이 좋아진다.